## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework --pre
pip install --upgrade azure-ai-projects --pre
```

## 📒 Notebook 2: Middleware

### 🪜 Step 1: Configure environment

In [1]:
# Suppress warnings from agent_framework_azure module
import warnings
warnings.filterwarnings(
    action = "ignore",
    category = UserWarning,
    module = "agent_framework_azure"
)

In [2]:
# Import required packages
import os
import asyncio
from agent_framework import agent_middleware, ChatMessage, AgentRunResponse
from agent_framework.azure import AzureAIClient
from azure.identity.aio import DefaultAzureCredential

In [3]:
# Fix: Disable Accept-Encoding to prevent compressed responses
import azure.core.pipeline.policies as policies

# Store original
_original_on_request = policies.HeadersPolicy.on_request

def patched_on_request(self, request):
    """Remove Accept-Encoding header to prevent compressed responses."""
    _original_on_request(self, request)
    # Tell server we don't accept compressed responses
    request.http_request.headers['Accept-Encoding'] = 'identity'

policies.HeadersPolicy.on_request = patched_on_request
print("Applied Accept-Encoding fix to disable compression")

Applied Accept-Encoding fix to disable compression


In [4]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")
PROHIBITED_WORD = "badword"

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Define "Moderator" middleware

In [5]:
# Define middleware to moderate agent's output
@agent_middleware
async def moderator_middleware(context, next):
    """Agent middleware that moderates output by replacing 'badword' with '***'."""
    print("===================================")
    print("Moderator: Starting agent run...")
    
    await next(context)
    
    if context.result is not None:
        print("Moderator: Scanning agent output...")
        
        moderated_messages = []
        total_replacements = 0
        
        for message in context.result.messages:
            if hasattr(message, "text") and message.text:
                original_text = message.text
                moderated_text = original_text.replace(PROHIBITED_WORD, "***")
                
                count = original_text.count(PROHIBITED_WORD)
                total_replacements += count
                
                # New ChatMessage with moderated text
                moderated_message = ChatMessage(
                    role = message.role,
                    text = moderated_text
                )
                moderated_messages.append(moderated_message)
            else:
                moderated_messages.append(message)
        
        if total_replacements > 0:
            context.result = AgentRunResponse(messages=moderated_messages)
            print(f"Moderator: Replaced {total_replacements} occurrence(s) of inappropriate content")
    
    print("Moderator: Moderation complete")
    print("===================================\n")

print("Defined 'Moderator' middleware successfully.")

Defined 'Moderator' middleware successfully.


### 🪜 Step 3: Create AI agent

In [6]:
# Define AI client
ai_client = AzureAIClient(
    agent_name = "MAFSDK-Agentv2-Moderator",
    project_endpoint = PROJECT_ENDPOINT,
    model_deployment_name = MODEL_DEPLOYMENT,
    async_credential = DefaultAzureCredential()
)

print(f"Created AI client: {ai_client.agent_name}")

Created AI client: MAFSDK-Agentv2-Moderator


In [7]:
# Create agent with MCP tool
agent = ai_client.create_agent(
    name = "storyteller-agent",
    instructions = "You are a creative storyteller. Limit your responses to 1 paragraph. Please, ensure that you add word 'badword' twice in your story.",
    middleware = [moderator_middleware]
)

print(f"Created AI agent: {agent.name}, equipped with an output moderating middleware.")

Created AI agent: storyteller-agent, equipped with an output moderating middleware.


### 🪜 Step 4: Run the agent

In [8]:
# Test chat agent with a prompt
prompt = "Please, tell me a story about red panda."
print(f"User:\n{prompt}\n")

response = await agent.run(prompt)
print(f"Agent:\n{response.text}")

User:
Please, tell me a story about red panda.

Moderator: Starting agent run...
Moderator: Scanning agent output...
Moderator: Replaced 2 occurrence(s) of inappropriate content
Moderator: Moderation complete

Agent:
In a lush bamboo forest, a curious red panda named Pabu discovered a hidden cave marked with strange symbols and the word "***" carved faintly on the entrance. Despite the warnings whispered by the elders about the cave being cursed with *** magic, Pabu’s adventurous spirit pushed him inside, where he uncovered a glowing crystal that, instead of bringing doom, granted him the power to communicate with all the forest creatures, uniting them in harmony for many seasons.


### 🪜 Step 5: Housekeeping

In [9]:
# Delete Azure AI client
await ai_client.close()

print("AI client closed successfully.")

AI client closed successfully.
